# FreshLens FL-2TC: Two-Tier Produce Classifier Fine-Tuning Pipeline

This interactive notebook provides an end-to-end environment to **fine-tune, monitor, and export** both models in the FreshLens Two-Tier CNN Pipeline:

1. **Tier 1 (Identity Classifier)**:
   - **Task**: 5-class classification (`banana`, `cucumber`, `eggplant`, `tomato`, `unknown`).
   - **Architecture**: `yolo11s-cls` (pretrained on ImageNet).
   - **Rejection Policy**: Confidence threshold $\ge 0.75$; below $0.75 \rightarrow$ `uncertain`.
   - **Augmentations**: Rotation, translation, mild scale, horizontal flip.

2. **Tier 2 (Freshness Classifier)**:
   - **Task**: 3-stage freshness categorization (`fresh`, `medium`, `spoiled`).
   - **Architecture**: `yolo11s-cls`.
   - **Rejection Policy**: Confidence threshold $\ge 0.50$.
   - **Crucial Rule (Color Preservation)**: Hue variation is strictly limited (`hsv_h=0.015`) because hue shifts corrupt ripeness and decay signals (e.g. green banana becoming yellow artificially).

3. **Convergence & Diagnostics**: Visualizes real-time loss and accuracy curves directly from Ultralytics logs.

## 1. Environment & Hardware Verification

In [ ]:
import os
import sys
import json
import shutil
from pathlib import Path
import matplotlib.pyplot as plt
import numpy as np
import pandas as pd
from PIL import Image
import torch
import ultralytics
from ultralytics import YOLO

print(f"Python:          {sys.version.split()[0]}")
print(f"PyTorch:         {torch.__version__}")
print(f"Ultralytics:     {ultralytics.__version__}")
cuda_available = torch.cuda.is_available()
print(f"CUDA Available:  {cuda_available}")
if cuda_available:
    device_name = torch.cuda.get_device_name(0)
    vram_gb = torch.cuda.get_device_properties(0).total_memory / (1024**3)
    print(f"GPU Device:      {device_name} ({vram_gb:.1f} GB VRAM)")
else:
    print("WARNING: CUDA not detected! Running on CPU will be significantly slower.")

## 2. Path Configuration & Dataset Inspection

In [ ]:
# Locate repository root automatically
cwd = Path.cwd()
if cwd.name == "notebooks":
    REPO_ROOT = cwd.parent.parent
elif (cwd / "packages" / "ml").is_dir():
    REPO_ROOT = cwd
else:
    REPO_ROOT = cwd.resolve()

DATA_DIR = REPO_ROOT / "data" / "ml-datasets" / "snapstock-fl2tc"
IDENTITY_DIR = DATA_DIR / "identity"
FRESHNESS_DIR = DATA_DIR / "freshness"
RUNS_DIR = REPO_ROOT / "runs"
WEIGHTS_DIR = REPO_ROOT / "packages" / "ml" / "weights"
BASE_MODEL = REPO_ROOT / "packages" / "ml" / "yolo11s-cls.pt"

print(f"Repo Root:      {REPO_ROOT.resolve()}")
print(f"Identity Data:  {IDENTITY_DIR.resolve()} (Exists: {IDENTITY_DIR.exists()})")
print(f"Freshness Data: {FRESHNESS_DIR.resolve()} (Exists: {FRESHNESS_DIR.exists()})")
print(f"Base Weights:   {BASE_MODEL.resolve()} (Exists: {BASE_MODEL.exists()})")

# Inspect dataset summary if present
summary_file = DATA_DIR / "dataset-summary.json"
if summary_file.exists():
    summary = json.loads(summary_file.read_text(encoding="utf-8"))
    print("\n--- DATASET SUMMARY ---")
    print(f"Total Unique Images: {summary.get('total_unique_samples')}")
    print("Identity Splits:", summary.get("identity_dataset", {}).get("splits", {}))
    print("Freshness Splits:", summary.get("freshness_dataset", {}).get("splits", {}))

## 3. Visual Preview of Training Samples

In [ ]:
def preview_dataset_samples(dataset_dir: Path, title: str, num_cols: int = 5):
    train_dir = dataset_dir / "train"
    if not train_dir.exists():
        print(f"Train directory not found: {train_dir}")
        return
    classes = [d.name for d in train_dir.iterdir() if d.is_dir()]
    fig, axes = plt.subplots(1, len(classes), figsize=(len(classes) * 3, 3))
    if len(classes) == 1:
        axes = [axes]
    for ax, cls in zip(axes, classes):
        cls_folder = train_dir / cls
        images = list(cls_folder.glob("*.jpg")) + list(cls_folder.glob("*.png"))
        if images:
            img = Image.open(images[0])
            ax.imshow(img)
            ax.set_title(f"{cls}\n(Total: {len(images)})")
        ax.axis("off")
    plt.suptitle(title, fontsize=13, y=1.05)
    plt.tight_layout()
    plt.show()

preview_dataset_samples(IDENTITY_DIR, "Tier 1: Identity Classes Preview")
preview_dataset_samples(FRESHNESS_DIR, "Tier 2: Freshness Classes Preview")

## 4. Fine-Tuning Tier 1: Identity Classifier

### Hyperparameters:
- **Backbone**: `yolo11s-cls`
- **Epochs**: 80 (with patience=15 early stopping)
- **Learning Rate**: `lr0=0.01` with cosine decay (`cos_lr=True`, `lrf=0.01`)
- **Augmentations**: Mild rotation (`degrees=15`), translation (`0.1`), scale (`0.15`), horizontal flip (`0.5`).
- **Label Smoothing**: `0.05` for regularizing hard boundaries.

In [ ]:
# Toggle between training from scratch or resuming
RESUME_TIER1 = False  # Set to True if resuming from existing last.pt

tier1_last = RUNS_DIR / "identity" / "identity-yolo11s-cls-v2" / "weights" / "last.pt"

if RESUME_TIER1 and tier1_last.exists():
    print(f"Resuming Tier 1 from {tier1_last}...")
    model_tier1 = YOLO(str(tier1_last))
    results_tier1 = model_tier1.train(resume=True)
else:
    print("Starting Tier 1 Fine-Tuning from baseline YOLO11s-cls...")
    base_weights = str(BASE_MODEL) if BASE_MODEL.exists() else "yolo11s-cls.pt"
    model_tier1 = YOLO(base_weights)
    
    # Run training
    results_tier1 = model_tier1.train(
        data=str(IDENTITY_DIR.resolve()),
        epochs=80,
        imgsz=224,
        batch=64,
        lr0=0.01,
        lrf=0.01,
        cos_lr=True,
        label_smoothing=0.05,
        patience=15,
        workers=2,
        device="0" if torch.cuda.is_available() else "cpu",
        seed=21,
        project=str((RUNS_DIR / "identity").resolve()),
        name="identity-yolo11s-cls-v2",
        exist_ok=True,
        plots=True,
        degrees=15,
        translate=0.1,
        scale=0.15,
        fliplr=0.5,
        flipud=0.0,
    )

## 5. Fine-Tuning Tier 2: Freshness Classifier

### Hyperparameters & Color Invariants:
- **Backbone**: `yolo11s-cls`
- **Epochs**: 70 (patience=15)
- **Initial LR**: `lr0=0.008` (slightly conservative fine-tuning rate)
- **Color Protection**: `hsv_h=0.015` (Hue change limited to 1.5% to ensure yellow/green/brown stages are preserved)
- **Texture Augmentation**: `hsv_s=0.7`, `hsv_v=0.4`, `degrees=15`, `translate=0.1`, `scale=0.1`.

In [ ]:
print("Starting Tier 2 (Freshness) Fine-Tuning...")
base_weights = str(BASE_MODEL) if BASE_MODEL.exists() else "yolo11s-cls.pt"
model_tier2 = YOLO(base_weights)

results_tier2 = model_tier2.train(
    data=str(FRESHNESS_DIR.resolve()),
    epochs=70,
    imgsz=224,
    batch=64,
    lr0=0.008,
    lrf=0.01,
    cos_lr=True,
    label_smoothing=0.05,
    patience=15,
    workers=2,
    device="0" if torch.cuda.is_available() else "cpu",
    seed=21,
    project=str((RUNS_DIR / "freshness").resolve()),
    name="freshness-yolo11s-cls-v2",
    exist_ok=True,
    plots=True,
    # Produce freshness-adapted augmentations
    degrees=15,
    translate=0.1,
    scale=0.1,
    fliplr=0.5,
    flipud=0.0,
    # Color preservation: restricted hue jitter
    hsv_h=0.015,
    hsv_s=0.7,
    hsv_v=0.4,
)

## 6. Training Curves & Convergence Diagnostics

Visualizes loss curves, Top-1 accuracy, and learning rate decay over epochs from `results.csv`.

In [ ]:
def plot_training_history(csv_path: Path, title: str):
    if not csv_path.exists():
        print(f"results.csv not found at: {csv_path}")
        return
    
    df = pd.read_csv(csv_path)
    df.columns = [c.strip() for c in df.columns]
    
    fig, (ax1, ax2, ax3) = plt.subplots(1, 3, figsize=(16, 4))
    
    # Loss
    if "train/loss" in df.columns:
        ax1.plot(df["epoch"], df["train/loss"], label="Train Loss", color="#2b5c8f", lw=2)
    if "val/loss" in df.columns:
        ax1.plot(df["epoch"], df["val/loss"], label="Val Loss", color="#d62728", lw=2)
    ax1.set_xlabel("Epoch")
    ax1.set_ylabel("Loss")
    ax1.set_title("Training & Validation Loss")
    ax1.legend()
    ax1.grid(True, linestyle="--", alpha=0.5)
    
    # Accuracy
    acc_col = "metrics/accuracy_top1" if "metrics/accuracy_top1" in df.columns else None
    if acc_col:
        ax2.plot(df["epoch"], df[acc_col], label="Top-1 Accuracy", color="#2ca02c", lw=2)
        peak_acc = df[acc_col].max()
        peak_ep = df.loc[df[acc_col].idxmax(), "epoch"]
        ax2.scatter([peak_ep], [peak_acc], color="red", zorder=5)
        ax2.annotate(f"Peak: {peak_acc:.2%} (Ep {int(peak_ep)})", (peak_ep, peak_acc),
                     textcoords="offset points", xytext=(-20, 10), fontweight="bold")
    ax2.set_xlabel("Epoch")
    ax2.set_ylabel("Top-1 Accuracy")
    ax2.set_title("Top-1 Validation Accuracy")
    ax2.grid(True, linestyle="--", alpha=0.5)
    ax2.legend()
    
    # Learning rate
    lr_cols = [c for c in df.columns if c.startswith("lr/")]
    if lr_cols:
        ax3.plot(df["epoch"], df[lr_cols[0]], label="Learning Rate", color="purple", lw=2)
    ax3.set_xlabel("Epoch")
    ax3.set_ylabel("LR")
    ax3.set_title("Cosine Annealing LR Decay")
    ax3.grid(True, linestyle="--", alpha=0.5)
    ax3.legend()
    
    plt.suptitle(title, fontsize=13, y=1.04)
    plt.tight_layout()
    plt.show()

# Plot Tier 1 Identity Curves
id_results = RUNS_DIR / "identity" / "identity-yolo11s-cls-v2" / "results.csv"
plot_training_history(id_results, "Tier 1 (Identity Classifier) Training Curves")

# Plot Tier 2 Freshness Curves
fresh_results = RUNS_DIR / "freshness" / "freshness-yolo11s-cls-v2" / "results.csv"
plot_training_history(fresh_results, "Tier 2 (Freshness Classifier) Training Curves")

## 7. Direct Evaluation on Held-Out Test Splits

Runs the unified `evaluate_fl2tc` module on the held-out test splits for both models.

In [ ]:
tier1_best = RUNS_DIR / "identity" / "identity-yolo11s-cls-v2" / "weights" / "best.pt"
tier2_best = RUNS_DIR / "freshness" / "freshness-yolo11s-cls-v2" / "weights" / "best.pt"

cmd = [
    sys.executable, "-m", "training.evaluate_fl2tc",
    "--dataset-dir", str(DATA_DIR),
    "--split", "test",
    "--device", "0" if torch.cuda.is_available() else "cpu",
    "--output-dir", str(RUNS_DIR / "eval")
]

if tier1_best.exists():
    cmd.extend(["--identity-weights", str(tier1_best)])
if tier2_best.exists():
    cmd.extend(["--freshness-weights", str(tier2_best)])

print("Executing evaluation:", " ".join(cmd))
import subprocess
eval_run = subprocess.run(cmd, cwd=str(REPO_ROOT / "packages" / "ml"), capture_output=True, text=True)
print(eval_run.stdout)
if eval_run.stderr:
    print("Stderr:", eval_run.stderr)

## 8. Export Weights for FreshLens API & ML Worker

Copies `best.pt` checkpoints into `packages/ml/weights/` and updates the deployment metadata.

In [ ]:
WEIGHTS_DIR.mkdir(parents=True, exist_ok=True)

exported = {}
if tier1_best.exists():
    dest_id = WEIGHTS_DIR / "identity-v2.pt"
    shutil.copy2(tier1_best, dest_id)
    print(f"Exported Model 1 to: {dest_id} ({dest_id.stat().st_size / (1024**2):.1f} MB)")
    exported["model1_identity"] = str(dest_id.name)

if tier2_best.exists():
    dest_fresh = WEIGHTS_DIR / "freshness-v2.pt"
    shutil.copy2(tier2_best, dest_fresh)
    print(f"Exported Model 2 to: {dest_fresh} ({dest_fresh.stat().st_size / (1024**2):.1f} MB)")
    exported["model2_freshness"] = str(dest_fresh.name)

# Write deployment metadata
deploy_meta = {
    "pipeline": "FL-2TC",
    "version": "v2",
    "exported_models": exported,
    "model1_classes": ["banana", "cucumber", "eggplant", "tomato", "unknown"],
    "model1_confidence_threshold": 0.75,
    "model2_classes": ["fresh", "medium", "spoiled"],
    "model2_confidence_threshold": 0.50,
}

meta_path = WEIGHTS_DIR / "model-metadata.json"
meta_path.write_text(json.dumps(deploy_meta, indent=2) + "\n", encoding="utf-8")
print(f"Updated deployment metadata: {meta_path}")
print(json.dumps(deploy_meta, indent=2))